# LangChain Chains
A **chain** connects multiple steps together — prompt → model → output parser — so you can build pipelines that process data step by step.

LangChain uses the `|` pipe operator to connect components, similar to Unix pipes.

In [1]:
pip install langchain langchain-ollama --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Simplest Chain — Prompt | Model

In [11]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(model='llama3.2')


prompt = ChatPromptTemplate.from_template('Tell me a fun fact about {topic}.')

# Chain: prompt → model
chain = prompt | llm 

response = chain.invoke({'topic': 'space'})
print(response.content)

Here's a fun fact about space:

Did you know that there is a giant storm on Jupiter that has been raging for at least 187 years? The Great Red Spot, as it's called, is a massive anticyclonic storm that is larger than Earth in diameter. It's a high-pressure region with clockwise rotation, and it's so stable that it's been continuously observed for centuries.

In fact, the Great Red Spot is so persistent that it's considered a "persistent anomaly" by scientists, and it's been the subject of numerous studies and observations over the years. Despite its long history, the Great Red Spot is still not fully understood, and scientists continue to study it to learn more about Jupiter's atmosphere and the behavior of massive storms in our solar system.

Isn't that out of this world?


## 2. Add Output Parser — Prompt | Model | Parser
Output parsers extract clean text (or structured data) from the model response.

In [7]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model='llama3.2')
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template('Translate "{text}" to {language}.')

# Chain: prompt → model → parser
chain = prompt | llm | parser

result = chain.invoke({'text': 'Hello, how are you?', 'language': 'French'})
print(type(result))   # str — clean string, not an AIMessage object
print(result)

<class 'langchain_core.messages.base.TextAccessor'>
"Bonjour, comment vas-tu?"


## 3. Sequential Chain — Output of One Step Feeds the Next

In [14]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model='llama3.2')
llm1 = ChatOllama(model='gemma3:4b')
parser = StrOutputParser()

# Step 1 — generate a short story
story_prompt = ChatPromptTemplate.from_template(
    'Write a 3-sentence story about {topic}.'
)

# Step 2 — summarise the story in one sentence
summary_prompt = ChatPromptTemplate.from_template(
    'Summarise this story in one sentence:\n{story}'
)

# Chain both steps together
chain = (
    story_prompt
    | llm
    | parser
    | (lambda story: {'story': story})   # pass output as input to next prompt
    | summary_prompt
    | llm1
    | parser
)

result = chain.invoke({'topic': 'a lost robot'})
print('Summary:', result)

Summary: A lost maintenance robot, Zeta-5, wanders through a deserted factory, relying on a faulty navigation system and a faint hum to guide it back to its charging station.


## 4. RunnableParallel — Run Multiple Chains at Once

In [9]:
from langchain_core.runnables import RunnableParallel
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model='llama3.2')
parser = StrOutputParser()

pros_chain = ChatPromptTemplate.from_template('List 3 pros of {technology}.') | llm | parser
cons_chain = ChatPromptTemplate.from_template('List 3 cons of {technology}.') | llm | parser

# Run both chains in parallel
parallel_chain = RunnableParallel(pros=pros_chain, cons=cons_chain)

result = parallel_chain.invoke({'technology': 'AI'})
print('PROS:\n', result['pros'])
print('\nCONS:\n', result['cons'])

PROS:
 Here are three pros of AI:

1. **Improved Efficiency and Productivity**: AI can automate repetitive and mundane tasks, freeing up human workers to focus on more complex and creative tasks. This can lead to significant increases in productivity and efficiency, particularly in industries such as customer service, finance, and healthcare.

2. **Enhanced Decision-Making and Analysis**: AI can process vast amounts of data quickly and accurately, making it an excellent tool for analyzing complex patterns and trends. This can help businesses and organizations make more informed decisions, identify new opportunities, and mitigate risks.

3. **Personalized Experiences and Services**: AI can be used to create personalized experiences and services that cater to individual preferences and needs. For example, AI-powered chatbots can provide customer support that is tailored to each user's specific queries and concerns. AI can also be used to create personalized product recommendations, impro

## 5. Chain with Streaming

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model='llama3.2')
chain = ChatPromptTemplate.from_template('Explain {topic} in simple terms.') | llm | StrOutputParser()

for chunk in chain.stream({'topic': 'neural networks'}):
    print(chunk, end='', flush=True)

In [13]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model='llama3.2')
llm1 = ChatOllama(model='gemma3:4b')

prompt = ChatPromptTemplate.from_template('Tell me a fun fact about {topic}.')

# Chain: prompt → model
chain = prompt | llm  | StrOutputParser()

response = chain.invoke({'topic': 'space'})
print(response)

Here's a fun fact about space:

Did you know that there is a giant storm on Jupiter that has been raging for at least 187 years? The Great Red Spot, as it's called, is a persistent anticyclonic storm on Jupiter, which means it's a high-pressure region with clockwise rotation. It's so large that three Earths could fit inside it, and it's still actively swirling to this day!

Isn't that out of this world?


## Summary

| Concept | Description |
|---|---|
| `prompt | llm` | Basic chain — fill prompt then call model |
| `prompt | llm | parser` | Add parser to get clean string output |
| Sequential chain | Output of one step becomes input of the next |
| `RunnableParallel` | Run multiple chains at the same time |
| `chain.stream()` | Stream output token by token |
| `chain.invoke()` | Run the full chain and return final result |